# Rework Impact Analysis in Laboratory Operations

## Objective
Evaluate whether exam rework is associated with:

- Higher turnaround time (TAT)
- Higher delay rates
- Increased operational risk

## Business Question
Does rework increase the probability of delayed exams?

In [ ]:
import pandas as pd

arquivo = pd.ExcelFile("dataset_excel.xlsx")

print(arquivo.sheet_names)

In [ ]:
df = pd.read_excel("dataset_excel.xlsx", sheet_name="Planilha1")

df.head()

## 1. Compare Average TAT by Rework Status

Hypothesis:
Exams requiring rework may have higher turnaround time.

In [ ]:
df.groupby("retrabalho")["tat_horas"].mean()

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

import matplotlib.pyplot as plt

grupos = ['Sem retrabalho','Com retrabalho']
tat = [7.99,11.19]

plt.bar(grupos, tat)

plt.title("Average Turnaround Time by Rework Status")
plt.ylabel("Horas")

for i,v in enumerate(tat):
    plt.text(i, v+0.2, str(v))

plt.savefig("outputs/tat_retrabalho.png")

plt.show()

## 2. Compare Delay Rates

Measure the proportion of delayed exams among:

- Exams without rework
- Exams with rework

In [ ]:
grupos = ['Sem retrabalho','Com retrabalho']
atraso = [23.8,53.1]

plt.bar(grupos, atraso)

plt.title("Delay Rate by Rework Status")
plt.ylabel("Taxa de atraso (%)")

for i,v in enumerate(atraso):
    plt.text(i, v+1, str(v)+"%")

plt.savefig("outputs/taxa_atraso.png")

plt.show()

## 3. Estimate Relative Risk

Calculate how much greater the delay risk is when rework occurs.

In [ ]:
53.1 / 23.8

## 4. Statistical Hypothesis Test

H0:
Delay is independent of rework.

H1:
Delay is associated with rework.

In [ ]:
import scipy.stats as stats

tabela = [
    [96,308],  #sem retrabalho
    [51,45]    #com retrabalho
]

chi2, p, dof, expected = stats.chi2_contingency(tabela)

print("Qui-quadrado:", chi2)
print("P-valor:", p)

In [ ]:
df["atrasado"] = (df["status_prazo"] == "Atrasado").astype(int)

df["atrasado"].head()

In [ ]:
df["retrabalho_bin"] = (df["retrabalho"] == "Sim").astype(int)

df[["retrabalho","retrabalho_bin"]].head()

## 5. Logistic Regression

Model:
Probability of delay as a function of rework.

In [ ]:
import statsmodels.api as sm

X = df["retrabalho_bin"]
X = sm.add_constant(X)

y = df["atrasado"]

modelo = sm.Logit(y, X).fit()

print(modelo.summary())

In [ ]:
import numpy as np

np.exp(1.2909)

## Conclusion

Findings suggest rework is an operational risk factor:

- Higher average TAT
- Higher delay rate
- Relative risk > 2x
- Odds of delay ~3.6x higher
- Statistical significance (p < 0.001)

Conclusion:
Rework appears associated with increased likelihood of SLA failure.